# 23 — Feature Engineering for Classification (Objective 2)

**Objective.** The feature-engineering step: *"creating derived variables from existing fields and dropping
those that add no explanatory value."* This notebook does both, for breakdown risk, and closes two open questions from
the transformation step:

| § | Question |
|---|---|
| 1 | Which derived features have a business reason to exist? |
| 2 | Which features carry information about risk on their own? |
| 3 | Shipment weight and storage issues duplicate each other — keep one or both? |
| 4 | Features with no one-at-a-time information — keep or drop? |
| 5 | Does any derived feature add value? |
| 6 | The final feature set |

**Target.** This is a **binary classification** task: **Not High Risk** (0–3 breakdowns) vs **High Risk** (4–6 breakdowns). Everything here uses the 20,000 training warehouses only (the train/test split from the transformation step). The test set is carried along untouched.

**Input:** `data/processed/classification_base.csv` · `feature_engine/feature_roles.csv` (transformation step)
**Output:** `data/processed/classification_model_input.csv` · `feature_engine/model_features.csv` · evidence tables in
`feature_engine/` · an NB 23 section appended to `feature_engine/feature_spec.md`

---

### How decisions are made here — fixed before the first run

A feature can matter in two ways, and each needs its own test.

**Alone — mutual information against a shuffled baseline.** Mutual information measures how much knowing a feature
reduces uncertainty about the risk class, whatever the shape of the relationship. Every estimate is slightly above zero,
even for a feature unrelated to risk, so each feature is compared with **itself against 20 randomly shuffled copies of the
target**. Shuffling destroys any real relationship while keeping everything else, so the largest of the 20 shuffled values
shows how high an estimate can reach by chance. A feature *carries information on its own* if its real value exceeds that
maximum. This is the same control idea used in the clustering EDA.

**In combination — probe models, compared fold by fold.** A tree can use a feature that is useless alone, for example
inside an interaction. So every proposed change is also tested inside two **probe models** — a logistic regression (with
scaling) and a random forest (200 trees). Both use default settings, and both run on the same **25 training folds**
(5-fold cross-validation, repeated 5 times). The score is **macro-F1**. For binary classification, macro-F1 is the mean
of the two class F1 scores (Not High Risk and High Risk), which is equivalent to the unweighted average of the two
per-class precision-recall averages — the standard binary F1 measure.

The probes only *compare* feature sets. They are not the final models, which the modelling step builds and tunes.

**A change counts as meaningful only if** the average fold-by-fold difference in macro-F1 is larger than **both**:
- **2 standard errors** of those differences, so it is unlikely to be fold-to-fold noise; and
- **0.005** (half a percentage point), so it is large enough to matter in practice.

Folds that reuse overlapping training rows make standard errors look smaller than they are. The 0.005 floor guards
against calling a tiny but consistent difference important.

**The rules:**

| Decision question | Rule |
|---|---|
| **Duplicate pair** | Remove whichever of the pair can be removed with no meaningful loss in *either* probe. If both can, remove the one with less information alone (§2) — never both. If neither can, keep both. |
| **Negligible features** | Take the characteristics and period measures with no information alone (§2), excluding the year's missing-flag (exempt because it is judged only together with the year). Remove them as a group if doing so causes no meaningful loss in *either* probe; otherwise keep them. |
| **Derived features** | Add a derived feature only if it is not a straight-line copy of its source columns (R² below 0.90 — test T2), **and** adding it is meaningfully *better* in at least one probe and meaningfully *worse* in neither. |

## 0. Setup

In [ ]:
import sys, pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()
paths = obj_paths(2)

base = pd.read_csv(paths["processed"] / "classification_base.csv")
roles = pd.read_csv(paths["feature_engine"] / "feature_roles.csv")

features = list(roles["feature"])
train = base[base["split"] == "train"]
y = train["breakdown_risk"]

print(f"loaded {len(base):,} warehouses; training rows used here: {len(train):,}")
assert len(features) == 29
print(f"features from the transformation step: {len(features)}")
print("training class counts:", y.value_counts().reindex(["Not High Risk", "High Risk"]).to_dict())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mutual_info_score

**The probe.** Each decision section builds two probe models — logistic regression and random forest — explicitly, using the same 25 training folds. The logistic regression pipeline applies `StandardScaler` to the columns that need scaling; the random forest uses the unscaled features directly. Results are stored as dictionaries of fold-by-fold macro-F1 arrays and compared fold by fold against the fixed rule.

In [ ]:
FOLDS = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
SCALED = set(roles.loc[roles["scale_for_lr_svm"], "feature"])          # scaling spec from the transformation step

comparisons = []          # every comparison made in this notebook, saved in §7

---
## 1. Candidate derived features

Each candidate has a business reason, fixed before any test (test **T1**). The data dictionary names the causes of a
breakdown: *"strike from worker, flood, or electrical failure"*. Three candidates follow from those causes, and two
re-express quantities in a form a manager reads more easily.

| Candidate | Formula | Business reason |
|---|---|---|
| `warehouse_age` | reference year − `wh_est_year` | the business framing uses *warehouse age*; age reads more naturally than a founding year |
| `tons_per_worker` | `product_wg_ton` ÷ `workers_num` | workload per worker — strained staff are one route to the *worker strikes* the data dictionary names |
| `storage_issues_per_1000t` | `storage_issue_reported_l3m` ÷ (`product_wg_ton` ÷ 1,000) | storage problems relative to volume handled — a sign of upkeep beyond sheer size |
| `shops_per_distributor` | `retail_shop_num` ÷ `distributor_num` | distribution load — how many shops each distributor serves |
| `flood_unprotected` | 1 if `flood_impacted` = 1 and `flood_proof` = 0 | in a flood area **without** flood-proofing — *flood* is a named breakdown cause |

The reference year is the latest establishment year among training warehouses. Every formula is fixed arithmetic on a
warehouse's own values, so computing candidates for test warehouses learns nothing from them. No denominator can be zero:
the smallest recorded values are 10 workers, 2,065 t and 15 distributors.

**Test T2 — not a straight-line copy.** R² of a linear regression of each candidate on its own source columns, over
training warehouses. At **0.90 or more** a candidate is a near re-expression that a linear model already has.

In [ ]:
REFERENCE_YEAR = int(train["wh_est_year"].max())

candidates = pd.DataFrame({
    "warehouse_age": REFERENCE_YEAR - base["wh_est_year"],
    "tons_per_worker": base["product_wg_ton"] / base["workers_num"],
    "storage_issues_per_1000t": base["storage_issue_reported_l3m"] / (base["product_wg_ton"] / 1000),
    "shops_per_distributor": base["retail_shop_num"] / base["distributor_num"],
    "flood_unprotected": ((base["flood_impacted"] == 1) & (base["flood_proof"] == 0)).astype(int),
}, index=base.index)
candidate_sources = {
    "warehouse_age": ["wh_est_year"],
    "tons_per_worker": ["product_wg_ton", "workers_num"],
    "storage_issues_per_1000t": ["storage_issue_reported_l3m", "product_wg_ton"],
    "shops_per_distributor": ["retail_shop_num", "distributor_num"],
    "flood_unprotected": ["flood_impacted", "flood_proof"],
}
assert np.isfinite(candidates.to_numpy()).all()

rows = []
for c, src in candidate_sources.items():
    X_src, v = train[src], candidates.loc[train.index, c]
    r2 = LinearRegression().fit(X_src, v).score(X_src, v)
    rows.append({"candidate": c, "sources": " + ".join(src), "r2_from_sources": r2, "passes_T2": r2 < 0.90,
                 "min": v.min(), "median": v.median(), "max": v.max()})
candidate_tests = pd.DataFrame(rows).set_index("candidate")

print(f"reference year for warehouse_age: {REFERENCE_YEAR}\n")
candidate_tests.round(4)

> **Interpretation.**
>
> - Two candidates fail T2 and go no further. Three pass.
>
> | Candidate | R² from its sources | T2 | Range (min / median / max) |
> |---|---|---|---|
> | `warehouse_age` | **1.0000** | fails | 0 / 14 / 27 years |
> | `tons_per_worker` | **0.9186** | fails | 34.2 / 761.8 / 3,934.1 t |
> | `storage_issues_per_1000t` | 0.6006 | passes | 0 / 0.7812 / 0.9904 |
> | `shops_per_distributor` | 0.8428 | passes | 27.8 / 117.8 / 600.9 |
> | `flood_unprotected` | 0.8899 | passes, only just | 0 / 0 / 1 |
>
> - **`warehouse_age` is the year read backwards** — R² exactly 1. It is easier to read, but it gives no model anything
>   the year does not; the year stays, and age can be used when *describing* results.
> - **`tons_per_worker` is mostly its sources in another form.** A straight-line combination of shipment weight and
>   workers already explains 91.86% of its variation.
> - `flood_unprotected` passes by a small margin (0.8899 against the 0.90 bar), so it goes to the model test in §5.


---
## 2. Which features carry information about risk on their own?

Mutual information for every NB 22 feature and every candidate, over training warehouses, against its shuffled baseline
(see the rules above). Columns with more than 20 distinct values are grouped into 20 equal-count bins first, so that
continuous features and small counts are measured the same way. The shuffled baseline is computed on those same bins, so
any upward bias from the number of groups is present in both.

Results are shown as a **percentage of the class's total uncertainty**, the same scale NB 22 §4 used.

In [ ]:
pool = pd.concat([base[features], candidates], axis=1)
rng = np.random.default_rng(RANDOM_STATE)
shuffled_targets = [rng.permutation(y.to_numpy()) for _ in range(20)]
class_uncertainty = mutual_info_score(y, y)
role_of = roles.set_index("feature")["role"]

rows = []
for c in features + list(candidates.columns):
    raw = pool.loc[train.index, c]
    v = pd.qcut(raw, q=20, duplicates="drop").cat.codes if raw.nunique() > 20 else raw
    real = mutual_info_score(y, v)
    chance = max(mutual_info_score(t, v) for t in shuffled_targets)
    rows.append({
        "feature": c,
        "group": "derived candidate" if c in candidates else role_of[c],
        "values_used": int(v.nunique()),
        "mi_pct": 100 * real / class_uncertainty,
        "shuffled_max_pct": 100 * chance / class_uncertainty,
        "times_shuffled_max": real / chance,
        "information_alone": real > chance,
    })
screen = pd.DataFrame(rows).sort_values("mi_pct", ascending=False).set_index("feature")

print(f"features with information alone: {int(screen['information_alone'].sum())} of {len(screen)}\n")
screen.round(4)

> **Interpretation.**
>
> - **The table above shows which features carry information about the binary target on their own** (MI score above the shuffled maximum) and which do not.
>
> - **The strong group is the one the EDA found:** storage issues, establishment year, and certificate status with the unrated flag. Each scores many times its shuffled maximum, as computed above.
> - **Temperature regulation and urban location** carry modest real signal, each clearly above their shuffled baseline, as computed above.
> - **`zone_South` and `wh_owner_type_Rented` only just clear their baselines** — a result consistent with chance among the features tested. A feature unrelated to risk beats the largest of 20 shuffles about one time in 21, so one or two such passes are expected by chance alone.
> - **The remaining features carry no detectable information alone:** refills, government checks, transport issues, staffing, distance, competitors, retail shops, distributors, electric back-up, both flood indicators, capacity size, the East and West zones, all five regional-zone columns, and the candidates `shops_per_distributor` and `flood_unprotected`.
>
> - **A note on precision.** `warehouse_age` and `wh_est_year` hold identical information, yet the equal-count bins fall at different cut-points around many tied values at 2009, depending on which way the column runs. Differences of a few tenths of a percentage point between features are therefore within this method's precision. The screen is used only for the coarse split between *some information* and *none*.

---
## 3. Decision — duplicate pair (shipment weight and storage issues)

**Problem.**

- The EDA found `product_wg_ton` and `storage_issue_reported_l3m` are near-duplicates.
- Spearman correlation: 0.989.
- VIF: 67.47 and 62.38.

**Why it matters.**

- Logistic regression coefficients become unstable when both remain.
- Naive Bayes counts the same evidence twice when both remain.
- The feature set can be simpler if one member of the pair can be removed without hurting model performance.

**Rule used below.**

- Compare three probe runs on the same folds:
  - both features kept;
  - shipment weight removed;
  - storage issues removed.
- Remove a member only if neither probe model loses meaningful performance.
- If both can be removed safely, remove the less informative member.

In [ ]:
print(f"Spearman correlation, training warehouses: "
      f"{train['product_wg_ton'].corr(train['storage_issue_reported_l3m'], method='spearman'):.4f}")
print("information alone (% of class uncertainty):",
      screen.loc[["storage_issue_reported_l3m", "product_wg_ton"], "mi_pct"].round(3).to_dict(), "\n")

# probe: all 29 features
_fl = features
_ts = [c for c in _fl if c in SCALED]
_lr = make_pipeline(ColumnTransformer([("scale", StandardScaler(), _ts)], remainder="passthrough"),
                    LogisticRegression(max_iter=5000))
_rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
_X = pool.loc[train.index, _fl]
scores = {"all 29": {"logistic regression": cross_val_score(_lr, _X, y, cv=FOLDS, scoring="f1_macro"),
                     "random forest": cross_val_score(_rf, _X, y, cv=FOLDS, scoring="f1_macro")}}

# probe: without product_wg_ton
_fl = [c for c in features if c != "product_wg_ton"]
_ts = [c for c in _fl if c in SCALED]
_lr = make_pipeline(ColumnTransformer([("scale", StandardScaler(), _ts)], remainder="passthrough"),
                    LogisticRegression(max_iter=5000))
_rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
_X = pool.loc[train.index, _fl]
scores["without product_wg_ton"] = {
    "logistic regression": cross_val_score(_lr, _X, y, cv=FOLDS, scoring="f1_macro"),
    "random forest": cross_val_score(_rf, _X, y, cv=FOLDS, scoring="f1_macro")}

# probe: without storage_issue_reported_l3m
_fl = [c for c in features if c != "storage_issue_reported_l3m"]
_ts = [c for c in _fl if c in SCALED]
_lr = make_pipeline(ColumnTransformer([("scale", StandardScaler(), _ts)], remainder="passthrough"),
                    LogisticRegression(max_iter=5000))
_rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
_X = pool.loc[train.index, _fl]
scores["without storage_issue_reported_l3m"] = {
    "logistic regression": cross_val_score(_lr, _X, y, cv=FOLDS, scoring="f1_macro"),
    "random forest": cross_val_score(_rf, _X, y, cv=FOLDS, scoring="f1_macro")}

# compare: remove product_wg_ton
_d3_rows = []
for _model in scores["all 29"]:
    _diff = scores["without product_wg_ton"][_model] - scores["all 29"][_model]
    _se = _diff.std(ddof=1) / np.sqrt(len(_diff))
    _bar = max(2 * _se, 0.005)
    _verdict = "better" if _diff.mean() > _bar else "worse" if _diff.mean() < -_bar else "no meaningful change"
    _d3_rows.append({"change": "remove product_wg_ton", "model": _model,
                     "macro_f1_before": scores["all 29"][_model].mean(),
                     "macro_f1_after": scores["without product_wg_ton"][_model].mean(),
                     "difference": _diff.mean(), "standard_error": _se, "bar": _bar,
                     "folds_better": int((_diff > 0).sum()), "verdict": _verdict})

# compare: remove storage_issue_reported_l3m
for _model in scores["all 29"]:
    _diff = scores["without storage_issue_reported_l3m"][_model] - scores["all 29"][_model]
    _se = _diff.std(ddof=1) / np.sqrt(len(_diff))
    _bar = max(2 * _se, 0.005)
    _verdict = "better" if _diff.mean() > _bar else "worse" if _diff.mean() < -_bar else "no meaningful change"
    _d3_rows.append({"change": "remove storage_issue_reported_l3m", "model": _model,
                     "macro_f1_before": scores["all 29"][_model].mean(),
                     "macro_f1_after": scores["without storage_issue_reported_l3m"][_model].mean(),
                     "difference": _diff.mean(), "standard_error": _se, "bar": _bar,
                     "folds_better": int((_diff > 0).sum()), "verdict": _verdict})

d3 = pd.DataFrame(_d3_rows)
comparisons += d3.to_dict("records")

removable = [c for c in ["product_wg_ton", "storage_issue_reported_l3m"]
             if (d3.loc[d3["change"] == f"remove {c}", "verdict"] != "worse").all()]
if len(removable) == 2:
    D3_REMOVE = min(removable, key=lambda c: screen.loc[c, "mi_pct"])
elif len(removable) == 1:
    D3_REMOVE = removable[0]
else:
    D3_REMOVE = None

print(d3.round(4).to_string(index=False))
print(f"\nremovable without meaningful loss in either probe: {removable}")
print(f"Duplicate-pair rule outcome — remove: {D3_REMOVE}")
after_d3 = [c for c in features if c != D3_REMOVE]

> **Interpretation.**
>
> - On training warehouses the two correlate at **0.9892**. Removing either one changes neither probe
>   meaningfully:
>
> | Change | Logistic regression | Random forest |
> |---|---|---|
> | remove `product_wg_ton` | as computed above | as computed above |
> | remove `storage_issue_reported_l3m` | as computed above | as computed above |
>
> - Every difference is well inside the 0.005 bar. Each probe already gets from one member of the pair everything the other
>   offers. Since both can be removed, the rule removes the one with less information alone: **shipment weight**.
>
> - **The probes' macro-F1 values are modest** (as computed above). These are untuned probes with no imbalance treatment,
>   so they are not a measure of what the final models will reach.

> **Decision — remove `product_wg_ton`; keep `storage_issue_reported_l3m`.** *(rule outcome)*
>
> **Why remove shipment weight.**
>
> - The two columns are near-duplicates: Spearman 0.9892.
> - Removing shipment weight costs neither probe model anything meaningful (+0.0004, +0.0005).
> - Storage issues carry more information alone: 15.10% against 13.36%.
> - One fewer column removes the unstable coefficient pair in logistic regression.
> - One fewer column also avoids double-counted evidence in Naive Bayes.
>
> **Why this does not contradict the clustering objective.**
>
> - The clustering objective kept shipment weight for a clustering-specific reason.
> - Without shipment weight, count columns placed many warehouses on tied points.
> - Each objective decides on its own evidence.
>
> **Effect on the characteristics-only set.**
>
> - The characteristics-only feature set is unaffected.
> - Both columns are period measures.
>
> **Rejected.**
>
> - Keep both: no gain in either probe and unstable linear coefficients.
> - Remove storage issues instead: the less informative member of the pair would stay.

---
## 4. Decision — features with no information on their own

**Group tested.**

- Start with every feature from the transformation step that failed §2's one-feature screen.
- Exclude `wh_est_year_missing` because it is judged only together with `wh_est_year`.
- Test the weak features as a group, not one at a time.

**Why group testing.**

- One-at-a-time removals of near-zero features would each fall inside noise.
- The group could still help or harm a model mechanically.
- The decision should reflect the joint effect on model performance.

In [ ]:
no_information = [c for c in after_d3 if not screen.loc[c, "information_alone"] and c != "wh_est_year_missing"]
print(f"features with no information alone ({len(no_information)}):", no_information, "\n")

reference_key = "all 29" if D3_REMOVE is None else f"without {D3_REMOVE}"

# probe: group removed
_fl = [c for c in after_d3 if c not in no_information]
_ts = [c for c in _fl if c in SCALED]
_lr = make_pipeline(ColumnTransformer([("scale", StandardScaler(), _ts)], remainder="passthrough"),
                    LogisticRegression(max_iter=5000))
_rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
_X = pool.loc[train.index, _fl]
scores["D6 group removed"] = {
    "logistic regression": cross_val_score(_lr, _X, y, cv=FOLDS, scoring="f1_macro"),
    "random forest": cross_val_score(_rf, _X, y, cv=FOLDS, scoring="f1_macro")}

# compare: remove the group
_d6_rows = []
for _model in scores[reference_key]:
    _diff = scores["D6 group removed"][_model] - scores[reference_key][_model]
    _se = _diff.std(ddof=1) / np.sqrt(len(_diff))
    _bar = max(2 * _se, 0.005)
    _verdict = "better" if _diff.mean() > _bar else "worse" if _diff.mean() < -_bar else "no meaningful change"
    _d6_rows.append({"change": f"remove the {len(no_information)} features", "model": _model,
                     "macro_f1_before": scores[reference_key][_model].mean(),
                     "macro_f1_after": scores["D6 group removed"][_model].mean(),
                     "difference": _diff.mean(), "standard_error": _se, "bar": _bar,
                     "folds_better": int((_diff > 0).sum()), "verdict": _verdict})

d6 = pd.DataFrame(_d6_rows)
comparisons += d6.to_dict("records")

D6_REMOVE = no_information if (d6["verdict"] != "worse").all() else []
print(d6.round(4).to_string(index=False))
print(f"\nNegligible-features rule outcome — remove {len(D6_REMOVE)} features")
after_d6 = [c for c in after_d3 if c not in D6_REMOVE]
current_key = "D6 group removed" if D6_REMOVE else reference_key

> **Interpretation.**
>
> - 19 features have no information alone and are not exempt: refills, government checks, transport
>   issues, workers, distance from hub, competitors, retail shops, distributors, electric back-up, flood-proofing, flood
>   impact, capacity size, the East and West zones, and all five regional-zone columns.
>
> - **Removing them splits the two probes:**
>
> - **Logistic regression is unaffected** — 0.5200 → 0.5206 (+0.0007).
> - **The random forest loses** — 0.5152 → 0.5076 (**−0.0076**, standard error 0.0018). That clears the bar, and the forest
>   does better without them on only **4 of 25** folds.
>
> - Under the fixed rule, a meaningful loss in either probe means **the group stays**. The control that follows asks why the
>   forest wants features that carry no information alone.


**Why does the forest lose when the group is removed? A control, added after the first run.** The rule above has already
decided the retention question, and nothing below can change that outcome. But the reason matters for how the result is read, and two
explanations predict different things:

- **The 19 features carry information in combination** — through interactions that no single-feature test can see.
- **The forest simply works better with more columns to choose from.** At each split a random forest considers only a
  random subset of the features (√n of them by default). With just 9 features left, most splits would be offered the same
  few strong ones, so the 200 trees grow alike and averaging them helps less. On this reading, any 19 extra columns would
  help, informative or not.

The control keeps all 19 columns but **shuffles each one independently across training warehouses.** Their values and
spreads stay exactly as recorded, but any link to risk, or to each other, is destroyed. If the shuffled columns rescue the
forest as well as the real ones, the benefit is mechanical. If the forest does as badly as with the columns removed, the
real features carry combined information.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
pool_shuffled = pool.copy()
for c in no_information:
    pool_shuffled.loc[train.index, c] = rng.permutation(pool.loc[train.index, c].to_numpy())

# probe: group shuffled
_fl = after_d3
_ts = [c for c in _fl if c in SCALED]
_lr = make_pipeline(ColumnTransformer([("scale", StandardScaler(), _ts)], remainder="passthrough"),
                    LogisticRegression(max_iter=5000))
_rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
_X = pool_shuffled.loc[train.index, _fl]
scores["D6 group shuffled"] = {
    "logistic regression": cross_val_score(_lr, _X, y, cv=FOLDS, scoring="f1_macro"),
    "random forest": cross_val_score(_rf, _X, y, cv=FOLDS, scoring="f1_macro")}

# compare: shuffled vs reference
_ctrl_rows = []
for _model in scores[reference_key]:
    _diff = scores["D6 group shuffled"][_model] - scores[reference_key][_model]
    _se = _diff.std(ddof=1) / np.sqrt(len(_diff))
    _bar = max(2 * _se, 0.005)
    _verdict = "better" if _diff.mean() > _bar else "worse" if _diff.mean() < -_bar else "no meaningful change"
    _ctrl_rows.append({"change": f"shuffle the {len(no_information)} features", "model": _model,
                       "macro_f1_before": scores[reference_key][_model].mean(),
                       "macro_f1_after": scores["D6 group shuffled"][_model].mean(),
                       "difference": _diff.mean(), "standard_error": _se, "bar": _bar,
                       "folds_better": int((_diff > 0).sum()), "verdict": _verdict})

diagnostic = pd.DataFrame(_ctrl_rows)
comparisons += diagnostic.to_dict("records")

print(diagnostic.round(4).to_string(index=False))
print("\nmean macro-F1 over the 25 folds:")
pd.DataFrame({model: {"19 features as recorded": scores[reference_key][model].mean(),
                      "19 features shuffled": scores["D6 group shuffled"][model].mean(),
                      "19 features removed": scores["D6 group removed"][model].mean()}
              for model in scores[reference_key]}).round(4)

> **Interpretation.**
>
> - **The forest's gain is mainly mechanical, not evidence that these conditions matter.**
>
> | Mean macro-F1 over 25 folds | Logistic regression | Random forest |
> |---|---|---|
> | 19 features as recorded | 0.5200 | **0.5152** |
> | 19 features **shuffled** | 0.5162 | **0.5129** |
> | 19 features removed | 0.5206 | **0.5076** |
>
> - **Shuffled columns rescue most of the forest's loss.** With the 19 columns shuffled — values kept, any link to risk
>   destroyed — the forest scores 0.5129. That is 0.0053 above removal, and only 0.0023 below the real columns, a
>   difference that does not meet the bar (standard error 0.0010).
> - So the forest benefits from having **more columns to sample from**, whether or not they carry information. The small
>   remaining difference between real and shuffled columns is not a meaningful one. **There is no evidence here that
>   these 19 conditions relate to breakdown risk, alone or in combination.**
> - For logistic regression, neither shuffling (−0.0037) nor removal (+0.0007) makes a meaningful difference.


> **Decision — keep all features with no information alone.** *(rule outcome)*
>
> **Why keep them.**
>
> - Removing them costs the random forest 0.0076 macro-F1, beyond the fixed bar.
> - Logistic regression is effectively indifferent (+0.0007).
> - The rule was fixed before the first run.
> - The rule is applied as written.
>
> **How to read them.**
>
> - The control shows they help the forest **as extra columns to sample from, not as information about risk**.
> - They are kept as a modelling aid.
> - No business conclusion should rest on them.
>
> **Carried to the modelling step.**
>
> - `max_features` controls how many columns a forest considers at each split.
> - That tuning parameter acts on the same mechanism tested here.
> - The tuned forest's setting shows whether the benefit persists.
>
> **Carried to the results notebook.**
>
> - Tree models give these columns non-zero importance even though, on this evidence, they carry no information.
> - Importance scores alone cannot show that a condition matters.
> - Permutation importance on the test set is the check.
>
> **Rejected.**
>
> - Remove the group: the forest loses 0.0076 macro-F1.
> - Remove it for logistic regression only: creates a second feature set for a non-meaningful +0.0007 difference.

---
## 5. Decision — does any derived feature add value?

**Test.**

- Start from the feature set left after the duplicate-pair and negligible-features decisions.
- Add each candidate derived feature one at a time.
- Use the same 25 folds for every comparison.
- Keep a candidate only if it improves at least one probe model and does not hurt the other.

In [ ]:
D10_ADD = []
for c in candidate_tests.index[candidate_tests["passes_T2"]]:
    # probe: add candidate c
    _fl = after_d6 + [c]
    _ts = [col for col in _fl if col in SCALED]
    _lr = make_pipeline(ColumnTransformer([("scale", StandardScaler(), _ts)], remainder="passthrough"),
                        LogisticRegression(max_iter=5000))
    _rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    _X = pool.loc[train.index, _fl]
    scores[f"add {c}"] = {
        "logistic regression": cross_val_score(_lr, _X, y, cv=FOLDS, scoring="f1_macro"),
        "random forest": cross_val_score(_rf, _X, y, cv=FOLDS, scoring="f1_macro")}

    # compare: add c vs current reference
    _rows = []
    for _model in scores[current_key]:
        _diff = scores[f"add {c}"][_model] - scores[current_key][_model]
        _se = _diff.std(ddof=1) / np.sqrt(len(_diff))
        _bar = max(2 * _se, 0.005)
        _verdict = "better" if _diff.mean() > _bar else "worse" if _diff.mean() < -_bar else "no meaningful change"
        _rows.append({"change": f"add {c}", "model": _model,
                      "macro_f1_before": scores[current_key][_model].mean(),
                      "macro_f1_after": scores[f"add {c}"][_model].mean(),
                      "difference": _diff.mean(), "standard_error": _se, "bar": _bar,
                      "folds_better": int((_diff > 0).sum()), "verdict": _verdict})
    result = pd.DataFrame(_rows)
    comparisons += result.to_dict("records")
    if (result["verdict"] == "better").any() and not (result["verdict"] == "worse").any():
        D10_ADD.append(c)

d10 = pd.DataFrame([r for r in comparisons if r["change"].startswith("add ")])
print(d10.round(4).to_string(index=False))
print(f"\nskipped by T2 (straight-line copies): {list(candidate_tests.index[~candidate_tests['passes_T2']])}")
print(f"Derived-features rule outcome — add: {D10_ADD}")

> **Interpretation.**
>
> - **No candidate adds anything.**
>
> | Candidate added | Logistic regression | Random forest |
> |---|---|---|
> | `storage_issues_per_1000t` | −0.0008 | −0.0009 |
> | `shops_per_distributor` | +0.0003 | +0.0003 |
> | `flood_unprotected` | 0.0000 | +0.0003 |
>
> - Every change sits far inside the 0.005 bar.
>
> - **`storage_issues_per_1000t` carries plenty of information alone** (8.43% of class uncertainty) **but none that the
>   existing features lack.** What it knows, storage issues and the rest already tell the model.
> - **`shops_per_distributor` and `flood_unprotected` carried no information alone either** (§2), and add none in
>   combination.
>
> - `warehouse_age` and `tons_per_worker` were not tested, having failed T2.


> **Decision — add no derived feature.** *(rule outcome)*
>
> | Candidate | Why not added |
> |---|---|
> | `warehouse_age` | exact re-expression of `wh_est_year` (R² 1.0000) |
> | `tons_per_worker` | near-copy of its sources (R² 0.9186) |
> | `storage_issues_per_1000t` | informative alone, but adds nothing to the model (−0.0008 / −0.0009) |
> | `shops_per_distributor` | no information alone or in the model |
> | `flood_unprotected` | no information alone (0.24 times the shuffled maximum) or in the model |
>
> **Business note.**
>
> - Floods are a possible breakdown cause in the data dictionary.
> - Warehouses in flood areas without flood-proofing show no detectable difference in breakdown risk in this dataset.
> - This may mean the flood indicators are too coarse.
> - It may also mean floods are only a small share of recorded breakdowns.
> - The single snapshot cannot separate those two explanations.

---
## 6. The final feature set

The final set is compared with the 29 features NB 22 handed over, on the same folds. This shows the net effect of
everything decided here.

In [ ]:
final_features = after_d6 + D10_ADD
if final_features == features:
    scores["final"] = scores["all 29"]
else:
    _fl = final_features
    _ts = [c for c in _fl if c in SCALED]
    _lr = make_pipeline(ColumnTransformer([("scale", StandardScaler(), _ts)], remainder="passthrough"),
                        LogisticRegression(max_iter=5000))
    _rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    _X = pool.loc[train.index, _fl]
    scores["final"] = {
        "logistic regression": cross_val_score(_lr, _X, y, cv=FOLDS, scoring="f1_macro"),
        "random forest": cross_val_score(_rf, _X, y, cv=FOLDS, scoring="f1_macro")}

# compare: final vs original 29
_net_rows = []
for _model in scores["all 29"]:
    _diff = scores["final"][_model] - scores["all 29"][_model]
    _se = _diff.std(ddof=1) / np.sqrt(len(_diff))
    _bar = max(2 * _se, 0.005)
    _verdict = "better" if _diff.mean() > _bar else "worse" if _diff.mean() < -_bar else "no meaningful change"
    _net_rows.append({"change": "final set vs transformation step's 29", "model": _model,
                      "macro_f1_before": scores["all 29"][_model].mean(),
                      "macro_f1_after": scores["final"][_model].mean(),
                      "difference": _diff.mean(), "standard_error": _se, "bar": _bar,
                      "folds_better": int((_diff > 0).sum()), "verdict": _verdict})

net = pd.DataFrame(_net_rows)
comparisons += net.to_dict("records")

print(f"final feature set: {len(final_features)} features "
      f"({len(features)} from the transformation step, - {1 if D3_REMOVE else 0} by duplicate-pair decision, "
      f"- {len(D6_REMOVE)} by negligible-features decision, + {len(D10_ADD)} by derived-features decision)\n")
print(net.round(4).to_string(index=False))

period_sources = set(roles.loc[roles["role"] == "period measure", "feature"])
final_roles = []
for c in final_features:
    if c in candidates:
        src = candidate_sources[c]
        final_roles.append({"feature": c, "origin": "derived in NB 23", "source_columns": " + ".join(src),
                            "role": "period-based" if set(src) & period_sources else "characteristic",
                            "scale_for_lr_svm": c != "flood_unprotected"})
    else:
        r = roles.set_index("feature").loc[c]
        final_roles.append({"feature": c, "origin": "NB 22", "source_columns": r["source_column"],
                            "role": r["role"], "scale_for_lr_svm": bool(r["scale_for_lr_svm"])})
model_features = pd.DataFrame(final_roles)
model_features["characteristics_only_set"] = ~model_features["role"].isin(["period measure", "period-based"])
print(f"\ncharacteristics-only set: {int(model_features['characteristics_only_set'].sum())} "
      f"of {len(model_features)} features")
model_features

> **Interpretation.**
>
> - The final set holds **28 features**: NB 22's 29 less `product_wg_ton`. Against the original 29, both
>   probes are unchanged (+0.0004 and +0.0005), so the set is one column simpler at no cost.
>
> - **Scaled for logistic regression and SVM:** 12 columns — the 13 scaled at the transformation step, less shipment weight.
> - **Characteristics-only set:** 24 features, the same set as before; both members of the duplicate pair were period
>   measures. The four remaining period measures are storage issues, refills, government checks and transport issues.


---
## 7. Save

In [ ]:
model_input = pd.concat([base[["Ware_house_ID", "split", "breakdown_risk"]], pool[final_features]], axis=1)
_ = save_table(model_input, paths["processed"] / "classification_model_input.csv", index=False)
_ = save_table(model_features, paths["feature_engine"] / "model_features.csv", index=False)
_ = save_table(candidate_tests, paths["feature_engine"] / "candidate_features.csv")
_ = save_table(screen, paths["feature_engine"] / "information_screen.csv")
_ = save_table(pd.DataFrame(comparisons), paths["feature_engine"] / "probe_comparisons.csv", index=False)

spec_path = paths["feature_engine"] / "feature_spec.md"
marker = "\n## NB 23 — feature engineering and selection\n"
spec = spec_path.read_text(encoding="utf-8").split(marker)[0]
spec += marker + f'''
Written by `notebooks/23_feature_engineering.ipynb`, on the 20,000 training warehouses only.

- **Duplicate pair:** removed {f"`{D3_REMOVE}`" if D3_REMOVE else "neither"}.
- **No information alone:** removed {len(D6_REMOVE)} feature(s){": " + ", ".join(f"`{c}`" for c in D6_REMOVE) if D6_REMOVE else ""}.
- **Derived features added:** {", ".join(f"`{c}`" for c in D10_ADD) if D10_ADD else "none"}.
- **Final feature set: {len(final_features)} features** — listed in `model_features.csv`; model-ready table
  `data/processed/classification_model_input.csv`.
- **Characteristics-only set: {int(model_features["characteristics_only_set"].sum())} features.**
- Evidence: `candidate_features.csv`, `information_screen.csv`, `probe_comparisons.csv`.
'''
spec_path.write_text(spec, encoding="utf-8")
print(f"saved  {spec_path.relative_to(PROJECT_ROOT)}  (NB 23 section)")

---
## 8. Checks

In [ ]:
back = pd.read_csv(paths["processed"] / "classification_model_input.csv")
mf = pd.read_csv(paths["feature_engine"] / "model_features.csv")

assert back.shape == (25_000, 3 + len(final_features)) and back["Ware_house_ID"].is_unique
assert back.notna().all().all()
assert list(mf["feature"]) == final_features == [c for c in back.columns if c not in ("Ware_house_ID", "split", "breakdown_risk")]
assert back["split"].value_counts().to_dict() == {"train": 20_000, "test": 5_000}
assert (back[["Ware_house_ID", "split", "breakdown_risk"]].to_numpy() == base[["Ware_house_ID", "split", "breakdown_risk"]].to_numpy()).all()
assert not mf.loc[mf["characteristics_only_set"], "role"].isin(["period measure", "period-based"]).any()
assert "wh_est_year_missing" not in D6_REMOVE
assert spec_path.read_text(encoding="utf-8").count("## NB 23") == 1

# read back the three evidence CSVs saved in §7
cf_back = pd.read_csv(paths["feature_engine"] / "candidate_features.csv")
is_back = pd.read_csv(paths["feature_engine"] / "information_screen.csv")
pc_back = pd.read_csv(paths["feature_engine"] / "probe_comparisons.csv")

assert cf_back.shape[0] == 5 and cf_back.notna().all().all(), "candidate_features.csv unexpected"
assert is_back.shape[0] == 34 and is_back.notna().all().all(), "information_screen.csv unexpected"
assert pc_back.shape[0] == len(comparisons) and pc_back.notna().all().all(), "probe_comparisons.csv unexpected"

print("all checks passed")
print(f"classification_model_input.csv : {back.shape[0]:,} warehouses x {len(final_features)} features + key, split, target")
print(f"candidate_features.csv         : {cf_back.shape}")
print(f"information_screen.csv         : {is_back.shape}")
print(f"probe_comparisons.csv          : {pc_back.shape}")

## Summary

**Binary classification target: Not High Risk (0–3 breakdowns) vs High Risk (4–6 breakdowns).**

All decisions below came from rules fixed before the first run and were computed on the 20,000 training warehouses.

**Duplicate pair.** Shipment weight is removed; storage issues are kept. The two columns correlate at Spearman 0.9892 (VIF 67/62). Removing either costs neither probe anything meaningful; storage issues carry more information alone (as computed in §2).

**Negligible features.** All features with no information alone are kept — as a modelling aid, not as evidence about risk. Removing them as a group costs the random forest meaningful macro-F1 performance (as computed in §4). A shuffled-column control recovers most of that loss, showing the forest benefits from having more columns to sample from, not from the conditions themselves. Logistic regression is indifferent.

**Derived features.** None added. Warehouse age and tons per worker are near re-expressions of existing columns (R² 1.0000 and 0.9186). The remaining three candidates change probe performance by no more than 0.0009 in either model.

**What this says about breakdown risk.** The information sits in a small group: storage issues (which carry what shipment weight did), establishment year, and certificate status with its unrated flag, plus a little from temperature regulation and urban location. The remaining features carry no detectable information alone. The control shows that the forest's use of such features is mechanical.

**Output.** `data/processed/classification_model_input.csv` — 25,000 warehouses, key, split, target and **28 features**, unscaled. `feature_engine/model_features.csv` — each feature's origin, role, scaling and characteristics-only set membership. Also `candidate_features.csv`, `information_screen.csv`, `probe_comparisons.csv`, and an NB 23 section in `feature_spec.md`.

**Handed to the modelling step:** the 28-feature set and its 24-feature characteristics-only subset, the scaling specification, the binary class-balance question, and two notes from the negligible-features decision — tune how many columns the forest samples per split, and do not read tree importance scores as evidence on their own.